In [ ]:
!pip install qiskit
!pip install qiskit-ibm-runtime
!pip install qiskit_aer
!pip install pylatexenc

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [ ]:
from qiskit import QuantumCircuit

qft2 = QuantumCircuit(2)
qft2.h(0)
qft2.cs(1,0)
qft2.h(1)
qft2.swap(0,1)
qft2.draw("mpl")

In [ ]:
import numpy as np

def qft(m):
    circ_qft = QuantumCircuit(m)
    for i in range(m):
        circ_qft.h(i) # Hadamard gate
        for j in range(i+1,m):
            circ_qft.cp(np.pi/2**(j-i),j,i) # Controlled-P gates
        circ_qft.barrier()
    # Final swaps
    for i in range(m//2):
        circ_qft.swap(i,m-i-1)
    return circ_qft

In [ ]:
qft4 = qft(4)
qft4.draw("mpl")

In [ ]:
def iqft(m):
    circ_iqft = QuantumCircuit(m)
    # Initial swaps
    for i in range(m//2-1,-1,-1):
        circ_iqft.swap(i,m-i-1)
    for i in range(m-1,-1,-1):
        for j in range(m-1,i,-1):
            circ_iqft.cp(-np.pi/2**(j-i),j,i) # Controlled-P gates
        circ_iqft.h(i) # Hadamard gate
        circ_iqft.barrier()

    return circ_iqft

In [ ]:
def invert(j,n): # Inverts integer j when represented with n bits
    str_j = bin(j)[2:] # Binary string for j without the leading "0b"
    str_j = (n-len(str_j))*'0' + str_j # Pad with initial zeroes up to n bits
    str_j = str_j[::-1] # Reverse the string
    return int(str_j,2) # Convert the string to a decimal integer

In [ ]:
from qiskit.circuit.library import UnitaryGate

def gate_mult(a,N):
    m = len(bin(N-1))-2 # Number of qubits needed to store numbers mod N
    M = 2**m

    matrix = []
    for i in range(M):
        matrix.append(M*[0]) # Initialize matrix to all zeroes

    for j in range(N):
        i = (a*j)%N # Position that we need to set to 1
        ii = invert(i,m) # Index of row for i according to Qiskit
        ij = invert(j,m) # Index of column for j according to Qiskit
        matrix[ii][ij] = 1

    for j in range(N,M):
        # Numbers bigger than N-1 are not considered, so we just leave them fixed
        ij = invert(j,m)
        matrix[ij][ij] = 1

    return UnitaryGate(matrix)

In [ ]:
def circuit_shor(a,N,m):
    # a is the number we want to obtain the period of
    # N is the number to factor
    # m is the number of qubits for the upper register

    n = len(bin(N-1))-2 # Number of qubits needed to store numbers mod N

    shor = QuantumCircuit(m+n,m)
    # Upper register is of size m
    # Lower register is of size n
    # We only measure the upper register, hence m classical bits

    shor.x(m+n-1) # Set the bottom register to |1>

    for i in range(m):
        shor.h(i) # Column of Hadamard's in the upper register

    for i in range(m):
        gate = gate_mult(a**(2**i),N).control(1) # Mult by a^(2^j) mod N
        shor.append(gate,[m-i-1]+list(range(m,m+n))) # Add controlled mult

    shor.barrier()

    shor.append(iqft(m),range(m)) # Inverse QFT

    shor.barrier()

    for i in range(m):
        shor.measure(i,i) # Measure the upper register

    return shor

In [ ]:
shor = circuit_shor(3,4,4)
shor.draw("mpl")

In [ ]:
shor = circuit_shor(2,15,8)

from qiskit_aer.primitives import SamplerV2 as Sampler
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = AerSimulator()
sampler = Sampler(seed = 1234)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
t_shor = pm.run(shor)
job = sampler.run([t_shor], shots = 100)

In [ ]:
results = job.result() # Access the results
d = results[0].data.c
res = d.get_counts()
for k in res:
    c = int(k[::-1],2) # Reverse the string and convert it to an integer
    print("Value:",c,"Frequency:",res[k])

In [ ]:
from fractions import Fraction

m=8
N=15
print(Fraction(64,2**m).limit_denominator(N))
print(Fraction(128,2**m).limit_denominator(N))
print(Fraction(192,2**m).limit_denominator(N))

print("2^4 mod 15 =",2**4%15)
print("2^2 -1 mod 15 =",2**2%15 -1)
print("2^2 +1 mod 15 =",2**2%15 + 1)

In [ ]:
shor = circuit_shor(16,21,8)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
t_shor = pm.run(shor)
job = sampler.run([t_shor], shots = 100)

results = job.result() # Access the results
d = results[0].data.c
res = d.get_counts()
for k in res:
    c = int(k[::-1],2) # Reverse the string a convert it to integer
    print("Value:",c,"Frequency:",res[k])

In [ ]:
m=8
N=21
print(Fraction(171,2**m).limit_denominator(N))
print(Fraction(85,2**m).limit_denominator(N))

print("16^3 mod 21 =",16**3%21)

In [ ]:
shor = circuit_shor(10,21,8)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
t_shor = pm.run(shor)
job = sampler.run([t_shor], shots = 100)

results = job.result() # Access the results
d = results[0].data.c
res = d.get_counts()
for k in res:
    c = int(k[::-1],2) # Reverse the string a convert it to integer
    print("Value:",c,"Frequency:",res[k])

In [ ]:
from fractions import Fraction
m=8
N=21
print(Fraction(43,2**m).limit_denominator(N))
print(Fraction(128,2**m).limit_denominator(N))
print(Fraction(171,2**m).limit_denominator(N))

In [ ]:
print("10^2 mod 21 =",10**2%21)
print("10^3 mod 21 =",10**3%21)
print("10^6 mod 21 =",10**6%21)

In [ ]:
from math import gcd

print(gcd(10**3%21-1,21))
print(gcd(10**3%21+1,21))